In [88]:
!pip uninstall -y delta-spark pyspark
!pip install pyspark==4.0.1 delta-spark==4.0.1

Found existing installation: delta-spark 4.0.1
Uninstalling delta-spark-4.0.1:
  Successfully uninstalled delta-spark-4.0.1
Found existing installation: pyspark 4.0.1
Uninstalling pyspark-4.0.1:
  Successfully uninstalled pyspark-4.0.1
  Using cached pyspark-4.0.1-py2.py3-none-any.whl
  Using cached delta_spark-4.0.1-py3-none-any.whl.metadata (1.9 kB)
Using cached delta_spark-4.0.1-py3-none-any.whl (43 kB)


# Customer & Order Management ETL Pipeline

### PySpark | Delta Lake | Data Quality | Bronze → Silver → Gold

This project demonstrates an end-to-end ETL pipeline for customer and
order data using PySpark and Delta Lake.

The pipeline implements Bronze, Silver, and Gold layers with data
cleaning, validation, referential integrity checks, business-rule
transformations, customer-level sales aggregation, and Delta Lake
persistence.

#*Project Overview*

# Customer Order Management — PySpark ETL Pipeline

## Project Overview

End-to-end data engineering pipeline built using PySpark and Delta Lake.

The project implements a Bronze → Silver → Gold architecture for processing customer and order data, including data cleaning, validation, transformation, business aggregation, and Delta Lake persistence.

### Technologies

- PySpark
- Delta Lake
- Python
- SQL concepts
- Databricks-style Medallion Architecture

#*Business Requirements*

## Business Requirements

The objective of this pipeline is to transform raw customer and order data into clean, validated, and analytics-ready datasets.

### Customer Requirements

- Validate and standardize Customer IDs.
- Clean and validate customer names.
- Standardize email addresses.
- Clean and validate phone numbers.
- Standardize city names.
- Convert registration dates into a consistent date format.
- Preserve raw values for data lineage and troubleshooting.

### Order Requirements

- Validate and standardize Order IDs.
- Validate Customer IDs.
- Convert order dates into a consistent date format.
- Clean and validate product names.
- Convert quantity into integer values and ensure quantity is greater than zero.
- Clean and validate unit prices.
- Standardize and validate order status.
- Preserve raw values for data lineage and troubleshooting.

### Gold Business Requirements

Create a customer-level sales summary containing:

- Number of orders
- Total amount
- Average order amount
- Total quantity

Cancelled orders are excluded from sales metrics.

Customers with no qualifying orders must remain in the Gold dataset with zero-valued sales metrics.

#*Architecture*

## ETL Architecture

This project follows the Medallion Architecture:


                RAW DATA
                   │
                   ▼
             ┌───────────┐
             │  BRONZE   │
             │           │
             │ Customers │
             │  Orders   │
             └─────┬─────┘
                   │
                   ▼
             ┌───────────┐
             │  SILVER   │
             │           │
             │ Cleaning  │
             │Validation │
             │  Raw Data │
             │  Retained │
             └─────┬─────┘
                   │
                   ▼
             ┌───────────┐
             │   GOLD    │
             │           │
             │ Customer  │
             │   Sales   │
             │  Summary  │
             └─────┬─────┘
                   │
                   ▼
             ┌───────────┐
             │   DELTA   │
             │   TABLE   │
             └───────────┘

#Layer Responsibilities

#Bronze

Stores raw source data.
Minimal transformation.
Preserves the original source values.

#Silver

Cleans and standardizes data.
Applies data-quality rules.
Creates validation flags.
Converts fields into appropriate data types.
Retains raw values for lineage.

#Gold

Applies business logic.
Excludes cancelled orders from sales metrics.
Aggregates order-level data to customer level.
Produces an analytics-ready customer sales summary.

#Delta Lake

Persists the Silver and Gold datasets.
Allows the transformed datasets to be read back and verified.

## Technology Stack

| Technology | Purpose |
|---|---|
| Python | Programming language |
| PySpark | Distributed data processing and ETL |
| Delta Lake | Reliable data storage and ACID transactions |
| Spark SQL | SQL-based data transformation concepts |
| Google Colab | Development environment |
| GitHub | Version control and portfolio hosting |

### Key PySpark Concepts Used

- DataFrame transformations
- `withColumn`
- `when` / `otherwise`
- String functions
- Regular expressions
- Date parsing
- Type casting
- `try_cast`
- Data validation
- Joins
- Aggregations
- `groupBy`
- `coalesce`
- Delta Lake read/write

## Data Quality & Validation Rules

The Silver layer applies field-level validation rules to identify invalid or malformed records.

### Customer Validation

| Field | Validation |
|---|---|
| Customer_ID | Must follow `CUST` + numeric format |
| Customer_Name | Alphabetic name with supported spaces, hyphens, and apostrophes |
| Email | Standardized and validated email format |
| Phone | Normalized to a 10-digit Indian mobile number |
| City | Standardized city name format |
| Registration_Date | Converted from supported input date formats |

### Order Validation

| Field | Validation |
|---|---|
| Order_ID | Must follow `ORD` + numeric format |
| Customer_ID | Must follow `CUST` + numeric format |
| Order_Date | Converted from supported input date formats |
| Product | Standardized product name |
| Quantity | Must be a positive integer |
| Unit_Price | Must be a positive numeric value |
| Order_Status | Must belong to the approved order-status list |

### Data Quality Approach

- Raw values are retained using `raw_*` columns.
- Validation results are stored using `valid_*` columns.
- Invalid cleaned values are set to `NULL`.
- `coalesce()` is used where appropriate to handle nullable validation results.
- Gold-level metrics are protected against `NULL` values using `coalesce()`.

## Gold Business Logic

The Gold layer creates a customer-level sales summary from the cleaned Silver Customers and Orders datasets.

### Transformation Steps

1. Perform a left join between Customers and Orders using `Customer_ID`.
2. Exclude orders with `Order_Status = 'CANCELLED'`.
3. Group the data by customer attributes.
4. Calculate the following business metrics:

| Metric | Business Logic |
|---|---|
| Number_of_Orders | Count of valid orders |
| Total_Amount | Sum of `Quantity × Unit_Price` |
| Average_Order_Amount | Average of `Quantity × Unit_Price` |
| Total_Quantity | Sum of ordered quantity |

### Handling Customers Without Orders

Customers without qualifying orders are retained in the Gold dataset.

Their sales metrics are represented as:

- `Number_of_Orders = 0`
- `Total_Amount = 0`
- `Average_Order_Amount = 0`
- `Total_Quantity = 0`

This ensures the Gold dataset provides a complete customer-level view rather than only returning customers who have made purchases.

## Pipeline Validation Results

The pipeline was validated at each major stage.

### Silver Layer

| Validation Check | Result |
|---|---:|
| Customer records | 10 |
| Order records | 11 |
| Duplicate Customer IDs | 0 |
| Duplicate Order IDs | 0 |

### Gold Layer

| Validation Check | Result |
|---|---:|
| Gold records | 10 |
| Negative order counts | 0 |
| Negative total amounts | 0 |
| Negative quantities | 0 |
| Gold Delta records after read-back | 10 |

### Persistence Validation

The Gold dataset was successfully:

1. Transformed using PySpark.
2. Written to Delta Lake.
3. Read back from Delta Lake.
4. Count-validated successfully with 10 records.

This confirms the end-to-end ETL and persistence workflow completed successfully.

## Project Outcome

This project demonstrates an end-to-end PySpark ETL pipeline using a Bronze → Silver → Gold architecture.

The pipeline successfully:

- Processes raw customer and order data.
- Applies data cleaning and standardization.
- Implements field-level data-quality validation.
- Preserves raw values for lineage and troubleshooting.
- Handles invalid data through validation flags and nullification.
- Performs customer-order joins and business aggregations.
- Excludes cancelled orders from sales metrics.
- Retains customers with no qualifying orders.
- Stores Silver and Gold datasets using Delta Lake.
- Validates the final Gold dataset after Delta read-back.

### Key Skills Demonstrated

**PySpark | ETL | Data Cleaning | Data Quality | Regular Expressions | Data Validation | Joins | Aggregations | Delta Lake | Medallion Architecture | Business Metrics**

#Step 1: Setting up the environment

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("UPI Payment Data Engineering")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

#Step 2: Create the Raw/Bronze DataFrames

#A. Customer raw data

In [90]:
customers_data = [
    (" cust1001 ", " john   doe ", " JOHN.DOE@GMAIL.COM ", "+91-9876543210", " silchar ", "2025-01-15"),
    ("CUST1002", "JANE smith", "jane.smith@gmail.com", "09876543210", "GUWAHATI", "2025/02/20"),
    (" cust1003", "  rahul   kumar", "RAHUL.KUMAR@YAHOO.COM", "98765 43210", "dibrugarh ", "15-03-2025"),
    ("CUST1004 ", "PRIYA SHARMA ", " priya.sharma@gmail.com", "+91 9123456789", " JORHAT", "2025-04-10"),
    ("CUST1005", "amit  das", "amit.das@gmail.com ", "9123456780", "silchar", "2025-05-05"),
    ("CUST1005", "amit das", "AMIT.DAS@GMAIL.COM", "9123456780", "SILCHAR", "2025-05-05"),
    ("CUST1006", " Neha Roy ", "neha.roy@gmail.com", "  9876543211 ", "Tezpur", "2025-06-12"),
    ("cust1007", "ARJUN SINGH", "arjun.singh@gmail.com", "+91-9988776655", " Guwahati ", "2025-07-18"),
    ("CUST1008", "  Meena Devi", "meena.devi@gmail.com", "99999-88888", "NAGAON", "2025-08-25"),
    ("CUST1009", "Ravi Kumar", "ravi.kumar@gmail", "9876543213", "Sivasagar", "2025-09-01"),
    ("CUST1010", "Sonia Das", "sonia.das@gmail.com", "9876543214", "Silchar", "2025-09-15"),
    (" CUST1011 ", "Kiran Paul", " kiran.paul@gmail.com ", "9876543215", "Haflong", "2025-10-03"),
    ("CUST1012", "Rohit Sen", "rohit.sen@gmail.com", "1234567890", "Guwahati", "2025-10-10"),
    ("CUSTOMER1013", "Anita Roy", "anita.roy@gmail.com", "9876543217", "Jorhat", "2025-11-01"),
    (None, "Missing ID", "missing.id@gmail.com", "9876543218", "Silchar", "2025-11-05")
]

customers_raw_df = spark.createDataFrame(
    customers_data,
    [
        "Customer_ID",
        "Customer_Name",
        "Email",
        "Phone",
        "City",
        "Registration_Date"
    ]
)

#B. Orders raw data

In [91]:
orders_data = [
    (" ord1001 ", "cust1001", "2026-01-05", " wireless   mouse ", "2", "₹1,500", " delivered "),
    ("ORD1002", "CUST1002", "2026/01/07", "Laptop", "1", "₹55,000", "DELIVERED"),
    ("ORD1003", " cust1003 ", "07-01-2026", " keyboard ", "3", "2500", " shipped "),
    ("ORD1004", "CUST1004", "2026-01-10", "Monitor", "2", "₹18,000", "confirmed"),
    ("ORD1005", "CUST1005", "2026-01-12", " USB Cable ", "5", "₹500", "Delivered"),
    ("ORD1005", "CUST1005", "2026-01-12", "USB Cable", "5", "500", "Delivered"),
    ("ORD1006", "cust1006", "2026-01-15", "Headphones", "1", "$120", "SHIPPED"),
    ("ORD1007", "CUST1007 ", "2026/01/18", " mobile phone ", "2", "₹25,000", "pending"),
    ("ORD1008", "CUST1008", "2026-01-20", "Tablet", "1", "₹20,000", "Returned"),
    ("ORD1009", "CUST9999", "2026-01-22", "Smart Watch", "2", "₹5,000", "Delivered"),
    ("ORD1010", "CUST1001", "2026-01-25", "Gaming Mouse", "0", "₹2,000", "Delivered"),
    ("ORD1011", "CUST1002", "2026-01-28", "Laptop Bag", "-1", "₹3,000", "Delivered"),
    ("ORD1012", "CUST1003", "2026-02-01", "Web Cam", "2", "₹4,000", "Unknown"),
    ("ORDER1013", "CUST1004", "2026-02-05", "Printer", "1", "₹12,000", "Delivered"),
    (None, "CUST1005", "2026-02-07", "Desk", "1", "₹8,000", "Pending"),
    ("ORD1015", "CUST1010", "2026-02-10", "Office Chair", "2", "₹7,500", "Cancelled"),
    ("ORD1016", "CUST1011", "2026-02-12", "Keyboard", "abc", "₹2,000", "Delivered"),
    ("ORD1017", "CUST1012", "2026-02-15", "Monitor", "1", "₹15,000", "Delivered"),
    ("ORD1018", "CUST1013", "2026-02-18", "Laptop", "1", "₹60,000", "Delivered"),
    ("ORD1019", "CUST1008", "bad-date", "Mouse", "2", "₹1,000", "Shipped")
]

orders_raw_df = spark.createDataFrame(
    orders_data,
    [
        "Order_ID",
        "Customer_ID",
        "Order_Date",
        "Product",
        "Quantity",
        "Unit_Price",
        "Order_Status"
    ]
)

#C. First Inspection

In [92]:
customers_raw_df.printSchema()
orders_raw_df.printSchema()

print("Customers:", customers_raw_df.count())
print("Orders:", orders_raw_df.count())

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Registration_Date: string (nullable = true)

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Unit_Price: string (nullable = true)
 |-- Order_Status: string (nullable = true)

Customers: 15
Orders: 20


#D. Creating Bronze Delta Files

#D.i. Bronze Delta Path

In [93]:
project_root = "/content/delta/Customer_Order_Management"

bronze_delta_cust_path = f"{project_root}/bronze/customers"
bronze_delta_ord_path = f"{project_root}/bronze/orders"

#D.ii. Loading customer and orders data into delta

In [94]:
customers_raw_df.write.format("delta").mode("overwrite").save(bronze_delta_cust_path)
orders_raw_df.write.format("delta").mode("overwrite").save(bronze_delta_ord_path)

#E. Extracting bronze data from delta

In [95]:
bronze_delta_customers_df = (spark
                             .read
                             .format("delta")
                             .load(bronze_delta_cust_path)
                             )

bronze_delta_orders_df = (spark
                          .read
                          .format("delta")
                          .load(bronze_delta_ord_path)
                          )

#Step 3: Silver Cleaning

#A. Customers

In [96]:
silver_customers_df = (bronze_delta_customers_df
                       .withColumn("raw_Customer_ID", sf.col("Customer_ID"))
                       .withColumn("Customer_ID", sf.upper(sf.trim("Customer_ID")))
                       .withColumn("valid_Customer_ID",
                                   sf.coalesce(
                                       sf.col("Customer_ID").rlike(r"^CUST\d+$"),
                                       sf.lit(False)))
                       .withColumn("Customer_ID",
                                   sf.when(sf.col("valid_Customer_ID"),
                                           sf.col("Customer_ID"))
                                   .otherwise(sf.lit(None)))
                       .withColumn("raw_Customer_Name", sf.col("Customer_Name"))
                       .withColumn("Customer_Name",
                                   sf.regexp_replace(
                                       sf.initcap(sf.trim("Customer_Name")), r"\s+", " "))
                       .withColumn("valid_Customer_Name",
                                   sf.coalesce(sf.col("Customer_Name")
                                   .rlike(r"^[A-Za-z]+(?:[-' ][A-Za-z]+)*$"), sf.lit(False)))
                       .withColumn("Customer_Name",
                                   sf.when(sf.col("valid_Customer_Name"),
                                           sf.col("Customer_Name"))
                                   .otherwise(sf.lit(None)))
                       .withColumn("raw_Email", sf.col("Email"))
                       .withColumn("Email",
                                   sf.regexp_replace(
                                       sf.lower(sf.trim("Email")), r"\s+", ""))
                       .withColumn("valid_Email",
                                   sf.coalesce(sf.col("Email")
                                   .rlike(r"^[a-z0-9]+(?:[-_\.][a-z0-9]+)?@[a-z]+(?:\.[a-z]+)+$"),
                                   sf.lit(False)))
                       .withColumn("Email",
                                   sf.when(sf.col("valid_Email"), sf.col("Email"))
                                   .otherwise(sf.lit(None)))
                       .withColumn("raw_Phone", sf.col("Phone"))
                       .withColumn("Phone",
                                   sf.regexp_replace(
                                       sf.regexp_replace(
                                           sf.trim("Phone"), r"[-\s]", ""), r"^(?:\+91|0)", ""))
                       .withColumn("valid_Phone",
                                   sf.coalesce(
                                       sf.col("Phone").rlike(r"^[6-9]\d{9}$"), sf.lit(False)))
                       .withColumn("Phone",
                                   sf.when(sf.col("valid_Phone"), sf.col("Phone"))
                                   .otherwise(sf.lit(None)))
                       .withColumn("raw_City", sf.col("City"))
                       .withColumn("City",
                                   sf.regexp_replace(
                                       sf.initcap(sf.trim("City")), r"\s+", " "))
                       .withColumn("valid_City",
                                   sf.coalesce(
                                       sf.col("City").rlike(r"^[A-Za-z]+(?:\s[A-Za-z]+)*$"),
                                       sf.lit(False)))
                       .withColumn("City",
                                   sf.when(sf.col("valid_City"), sf.col("City"))
                                   .otherwise(sf.lit(None)))
                       .withColumn("raw_Registration_Date", sf.col("Registration_Date"))
                       .withColumn("Registration_Date", sf.trim("Registration_Date"))
                       .withColumn("Registration_Date",
                                   sf.coalesce(
                                       sf.try_to_timestamp("Registration_Date",
                                                           sf.lit("yyyy-MM-dd"))
                                       .cast("date"),
                                       sf.try_to_timestamp("Registration_Date",
                                                           sf.lit("yyyy/MM/dd"))
                                       .cast("date"),
                                       sf.try_to_timestamp("Registration_Date",
                                                           sf.lit("dd-MM-yyyy"))
                                       .cast("date")))
                       .withColumn("valid_Registration_Date",
                                   sf.col("Registration_Date").isNotNull())
                       )

#B. Orders

In [97]:
silver_orders_df = (bronze_delta_orders_df
                    .withColumn("raw_Order_ID", sf.col("Order_ID"))
                    .withColumn("Order_ID",
                                sf.upper(sf.trim("Order_ID")))
                    .withColumn("valid_Order_ID",
                                sf.coalesce(
                                    sf.col("Order_ID").rlike(r"^ORD\d+$"),
                                    sf.lit(False)))
                    .withColumn("Order_ID",
                                sf.when(sf.col("valid_Order_ID"),
                                        sf.col("Order_ID"))
                                .otherwise(sf.lit(None)))
                    .withColumn("raw_Customer_ID", sf.col("Customer_ID"))
                    .withColumn("Customer_ID",
                                sf.upper(sf.trim("Customer_ID")))
                    .withColumn("valid_Customer_ID",
                                sf.coalesce(
                                    sf.col("Customer_ID").rlike(r"^CUST\d+$"),
                                    sf.lit(False)))
                    .withColumn("Customer_ID",
                                sf.when(sf.col("valid_Customer_ID"),
                                        sf.col("Customer_ID"))
                                .otherwise(sf.lit(None)))
                    .withColumn("raw_Order_Date", sf.col("Order_Date"))
                    .withColumn("Order_Date", sf.trim("Order_Date"))
                    .withColumn("Order_Date",
                                sf.coalesce(
                                    sf.try_to_timestamp("Order_Date", sf.lit("yyyy-MM-dd"))
                                    .cast("date"),
                                    sf.try_to_timestamp("Order_Date", sf.lit("dd-MM-yyyy"))
                                    .cast("date"),
                                    sf.try_to_timestamp("Order_Date", sf.lit("yyyy/MM/dd"))
                                    .cast("date")))
                    .withColumn("valid_Order_Date",
                                sf.col("Order_Date").isNotNull())
                    .withColumn("raw_Product", sf.col("Product"))
                    .withColumn("Product",
                                sf.regexp_replace(
                                    sf.initcap(sf.trim("Product")), r"\s+", " "))
                    .withColumn("valid_Product",
                                sf.coalesce(
                                    sf.col("Product").rlike(r"^[A-Za-z]+(?:\s[A-Za-z]+)*$"),
                                    sf.lit(False)))
                    .withColumn("Product",
                                sf.when(sf.col("valid_Product"), sf.col("Product"))
                                .otherwise(sf.lit(None)))
                    .withColumn("raw_Quantity", sf.col("Quantity"))
                    .withColumn("Quantity",
                                sf.trim("Quantity").try_cast("int"))
                    .withColumn("valid_Quantity",
                                sf.coalesce(
                                    sf.col("Quantity").isNotNull() &
                                     (sf.col("Quantity") > 0),
                                    sf.lit(False)))
                    .withColumn("Quantity",
                                sf.when(sf.col("valid_Quantity"), sf.col("Quantity"))
                                .otherwise(sf.lit(None)))
                    .withColumn("raw_Unit_Price", sf.col("Unit_Price"))
                    .withColumn("Unit_Price",
                                sf.regexp_replace(
                                    sf.trim("Unit_Price"), r"(?:[$₹,])", "")
                                .try_cast("double"))
                    .withColumn("valid_Unit_Price",
                                sf.coalesce(
                                    sf.col("Unit_Price").isNotNull() &
                                     (sf.col("Unit_Price") > 0),
                                    sf.lit(False)))
                    .withColumn("Unit_Price",
                                sf.when(sf.col("valid_Unit_Price"), sf.col("Unit_Price"))
                                .otherwise(sf.lit(None)))
                    .withColumn("raw_Order_Status", sf.col("Order_Status"))
                    .withColumn("Order_Status",
                                sf.upper(sf.trim("Order_Status")))
                    .withColumn("valid_Order_Status",
                                sf.coalesce(
                                    sf.col("Order_Status").isin("PENDING", "CONFIRMED",
                                                                "SHIPPED", "DELIVERED",
                                                                "CANCELLED"),
                                    sf.lit(False)))
                    .withColumn("Order_Status",
                                sf.when(sf.col("valid_Order_Status"), sf.col("Order_Status"))
                                .otherwise(sf.lit(None)))
                    )

#C. Validation of customers and orders table

In [98]:
print("Silver Customers records", silver_customers_df.count())

print("Customers", (silver_customers_df.filter(sf.col("valid_Customer_ID"))
                    .groupBy("Customer_ID")
                    .count()
                    .filter(sf.col("count") > 1)
                    .count()))

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.13/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/socket.py", line 723, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
(silver_customers_df
 .filter(sf.col("valid_Customer_ID") == False)
 .select("raw_Customer_ID")
 .show())

(silver_customers_df
 .filter(sf.col("valid_Customer_Name") == False)
 .select("raw_Customer_Name")
 .show())

(silver_customers_df
 .filter(sf.col("valid_Email") == False)
 .select("raw_Email")
 .show())

(silver_customers_df
 .filter(sf.col("valid_Phone") == False)
 .select("raw_Phone")
 .show())

(silver_customers_df
 .filter(sf.col("valid_City") == False)
 .select("raw_City")
 .show())

(silver_customers_df
 .filter(sf.col("valid_Registration_Date") == False)
 .select("raw_Registration_Date")
 .show())

In [ ]:
print("Silver Orders records", silver_orders_df.count())

print("Orders", (silver_orders_df.filter(sf.col("valid_Order_ID"))
                    .groupBy("Order_ID")
                    .count()
                    .filter(sf.col("count") > 1)
                    .count()))

In [ ]:
(silver_orders_df
 .filter(sf.col("valid_Order_ID") == False)
 .select("raw_Order_ID")
 .show())

(silver_orders_df
 .filter(sf.col("valid_Customer_ID") == False)
 .select("raw_Customer_ID")
 .show())

(silver_orders_df
 .filter(sf.col("valid_Order_Date") == False)
 .select("raw_Order_Date")
 .show())

(silver_orders_df
 .filter(sf.col("valid_Product") == False)
 .select("raw_Product")
 .show())

(silver_orders_df
 .filter(sf.col("valid_Quantity") == False)
 .select("raw_Quantity")
 .show())

(silver_orders_df
 .filter(sf.col("valid_Unit_Price") == False)
 .select("raw_Unit_Price")
 .show())

(silver_orders_df
 .filter(sf.col("valid_Order_Status") == False)
 .select("raw_Order_Status")
 .show())

#D. Checking invalid values become NULL when cleaned

In [ ]:
(silver_customers_df
 .filter((~sf.col("valid_Customer_ID")) &
         (sf.col("Customer_ID").isNotNull()))
 .count()
 )

(silver_customers_df
 .filter((~sf.col("valid_Customer_Name")) &
         (sf.col("Customer_Name").isNotNull()))
 .count()
 )

(silver_customers_df
 .filter((~sf.col("valid_Email")) &
         (sf.col("Email").isNotNull()))
 .count()
 )

(silver_customers_df
 .filter((~sf.col("valid_Phone")) &
         (sf.col("Phone").isNotNull()))
 .count()
 )

(silver_customers_df
 .filter((~sf.col("valid_City")) &
         (sf.col("City").isNotNull()))
 .count()
 )

(silver_customers_df
 .filter((~sf.col("valid_Registration_Date")) &
         (sf.col("Registration_Date").isNotNull()))
 .count()
 )

In [ ]:
(silver_orders_df
 .filter((~sf.col("valid_Order_ID")) &
         (sf.col("Order_ID").isNotNull()))
 .count()
 )

(silver_orders_df
 .filter((~sf.col("valid_Customer_ID")) &
         (sf.col("Customer_ID").isNotNull()))
 .count()
 )

(silver_orders_df
 .filter((~sf.col("valid_Order_Date")) &
         (sf.col("Order_Date").isNotNull()))
 .count()
 )

(silver_orders_df
 .filter((~sf.col("valid_Product")) &
         (sf.col("Product").isNotNull()))
 .count()
 )

(silver_orders_df
 .filter((~sf.col("valid_Quantity")) &
         (sf.col("Quantity").isNotNull()))
 .count()
 )

(silver_orders_df
 .filter((~sf.col("valid_Unit_Price")) &
         (sf.col("Unit_Price").isNotNull()))
 .count()
 )

(silver_orders_df
 .filter((~sf.col("valid_Order_Status")) &
         (sf.col("Order_Status").isNotNull()))
 .count()
 )

#E. Establishing referrential integrity

In [ ]:
(silver_orders_df
 .filter(sf.col("valid_Customer_ID"))
 .join(silver_customers_df.select("Customer_ID"),
       on= "Customer_ID", how="left_anti")
 .select("Customer_ID")
 .show(truncate=False)
 )

#F. Final Silver Validation: Raw-Value Preservation

In [ ]:
(silver_customers_df
 .filter(~sf.col("valid_Customer_ID") &
         sf.col("raw_Customer_ID").isNotNull())
 .show()
 )

(silver_customers_df
 .filter(~sf.col("valid_Customer_Name") &
         sf.col("raw_Customer_Name").isNotNull())
 .show()
 )

(silver_customers_df
 .filter(~sf.col("valid_Email") &
         sf.col("raw_Email").isNotNull())
 .show()
 )

(silver_customers_df
 .filter(~sf.col("valid_Phone") &
         sf.col("raw_Phone").isNotNull())
 .show()
 )

(silver_customers_df
 .filter(~sf.col("valid_City") &
         sf.col("raw_City").isNotNull())
 .show()
 )

(silver_customers_df
 .filter(~sf.col("valid_Registration_Date") &
         sf.col("raw_Registration_Date").isNotNull())
 .show()
 )

#Step 4: Building Quarantine Table

#A. Flagging invalid records

#i. Customers invalid records

In [ ]:
silver_customers_df.printSchema()

In [ ]:
silver_customers_df = (silver_customers_df
                       .withColumn("Validation_Failed",
                                   ~(sf.col("valid_Customer_ID") &
                                     sf.col("valid_Customer_Name") &
                                     sf.col("valid_Email") &
                                     sf.col("valid_Phone") &
                                     sf.col("valid_City") &
                                     sf.col("valid_Registration_Date")))
                       )

In [ ]:
silver_customers_df.show()

#ii. Orders invalid records

In [ ]:
silver_orders_df.printSchema()

In [ ]:
silver_orders_df = (silver_orders_df
                    .withColumn("Validation_Failed",
                                ~(sf.col("valid_Order_ID") &
                                  sf.col("valid_Customer_ID") &
                                  sf.col("valid_Order_Date") &
                                  sf.col("valid_Product") &
                                  sf.col("valid_Quantity") &
                                  sf.col("valid_Unit_Price") &
                                  sf.col("valid_Order_Status")))
                    )

In [ ]:
silver_orders_df.show()

#B. Creating Quarantine tables

#I. Quarantined Customers table

#i. Quarantine customer_id

In [ ]:
quarantine_customer_id_df = (silver_customers_df
                             .filter(~sf.col("valid_Customer_ID"))
                             .select(sf.col("raw_Customer_ID").alias("Customer_ID"),
                                     sf.lit("Customer_ID").alias("Column_Name"),
                                     sf.col("raw_Customer_ID").alias("Invalid_Value"),
                                     sf.lit("DQ001").alias("Rule_ID"),
                                     sf.lit("Invalid Customer_ID format")
                                     .alias("Invalid_Reason"))
                             )

In [ ]:
quarantine_customer_id_df.show()

#ii. Quarantine Customer_Name

In [ ]:
quarantine_customer_name_df = (silver_customers_df
                             .filter(~sf.col("valid_customer_Name"))
                             .select(sf.col("Customer_ID").alias("Customer_ID"),
                                     sf.lit("Customer_Name").alias("Column_Name"),
                                     sf.col("raw_Customer_Name").alias("Invalid_Value"),
                                     sf.lit("DQ002").alias("Rule_ID"),
                                     sf.lit("Invalid Customer_Name format")
                                     .alias("Invalid_Reason"))
                             )

In [ ]:
quarantine_customer_name_df.show()

#iii. Quarantine Email

In [ ]:
quarantine_email_df = (silver_customers_df
                             .filter(~sf.col("valid_Email"))
                             .select(sf.col("Customer_ID").alias("Customer_ID"),
                                     sf.lit("Email").alias("Column_Name"),
                                     sf.col("raw_Email").alias("Invalid_Value"),
                                     sf.lit("DQ003").alias("Rule_ID"),
                                     sf.lit("Invalid Email format")
                                     .alias("Invalid_Reason"))
                             )

In [ ]:
quarantine_email_df.show()

#iv. Quarantine Phone

In [ ]:
quarantine_phone_df = (silver_customers_df
                             .filter(~sf.col("valid_Phone"))
                             .select(sf.col("Customer_ID").alias("Customer_ID"),
                                     sf.lit("Phone").alias("Column_Name"),
                                     sf.col("raw_Phone").alias("Invalid_Value"),
                                     sf.lit("DQ004").alias("Rule_ID"),
                                     sf.lit("Invalid Phone format")
                                     .alias("Invalid_Reason"))
                             )

In [ ]:
quarantine_phone_df.show()

#v. Quarantine City

In [ ]:
quarantine_city_df = (silver_customers_df
                             .filter(~sf.col("valid_City"))
                             .select(sf.col("Customer_ID").alias("Customer_ID"),
                                     sf.lit("City").alias("Column_Name"),
                                     sf.col("raw_City").alias("Invalid_Value"),
                                     sf.lit("DQ005").alias("Rule_ID"),
                                     sf.lit("Invalid City format")
                                     .alias("Invalid_Reason"))
                             )

In [ ]:
quarantine_city_df.show()

#vi. Quarantine Registration_Date

In [ ]:
quarantine_registration_date_df = (silver_customers_df
                             .filter(~sf.col("valid_Registration_Date"))
                             .select(sf.col("Customer_ID").alias("Customer_ID"),
                                     sf.lit("Registration_Date").alias("Column_Name"),
                                     sf.col("raw_Registration_Date").alias("Invalid_Value"),
                                     sf.lit("DQ006").alias("Rule_ID"),
                                     sf.lit("Invalid Registration_Date format")
                                     .alias("Invalid_Reason"))
                             )

In [ ]:
quarantine_registration_date_df.show()

#vii. Combined Quarantine Table

In [ ]:
quarantine_customers_df = (quarantine_customer_id_df
                           .unionByName(quarantine_customer_name_df)
                           .unionByName(quarantine_email_df)
                           .unionByName(quarantine_phone_df)
                           .unionByName(quarantine_city_df)
                           .unionByName(quarantine_registration_date_df)
)

In [ ]:
quarantine_customers_df.show()

#II. Quarantined Orders table

#i. Quarantine Order_ID

In [ ]:
quarantine_order_id_df = (silver_orders_df
                          .filter(~sf.col("valid_Order_ID"))
                          .select(sf.col("raw_Order_ID").alias("Order_ID"),
                                  sf.lit("Order_ID").alias("Column_Name"),
                                  sf.col("raw_Order_ID").alias("Invalid_Value"),
                                  sf.lit("DQ011").alias("Rule_ID"),
                                  sf.lit("Invalid Order_ID format").alias("Invalid_Reason"))
                          )

In [ ]:
quarantine_order_id_df.show()

#ii. Quarantine Customer_ID

In [ ]:
quarantine_order_customers_df = (silver_orders_df
                          .filter(~sf.col("valid_Customer_ID"))
                          .select(sf.col("Order_ID").alias("Order_ID"),
                                  sf.lit("Customer_ID").alias("Column_Name"),
                                  sf.col("raw_Customer_ID").alias("Invalid_Value"),
                                  sf.lit("DQ012").alias("Rule_ID"),
                                  sf.lit("Invalid Customer_ID format").alias("Invalid_Reason"))
                          )

In [ ]:
quarantine_order_customers_df.show()

#iii. Quarantine Order_Date

In [ ]:
quarantine_order_date_df = (silver_orders_df
                          .filter(~sf.col("valid_Order_Date"))
                          .select(sf.col("Order_ID").alias("Order_ID"),
                                  sf.lit("Order_Date").alias("Column_Name"),
                                  sf.col("raw_Order_Date").alias("Invalid_Value"),
                                  sf.lit("DQ013").alias("Rule_ID"),
                                  sf.lit("Invalid Order_Date format").alias("Invalid_Reason"))
                          )

In [ ]:
quarantine_order_date_df.show()

#iv. Quarantine Product

In [ ]:
quarantine_product_df = (silver_orders_df
                          .filter(~sf.col("valid_Product"))
                          .select(sf.col("Order_ID").alias("Order_ID"),
                                  sf.lit("Product").alias("Column_Name"),
                                  sf.col("raw_Product").alias("Invalid_Value"),
                                  sf.lit("DQ014").alias("Rule_ID"),
                                  sf.lit("Invalid Product format").alias("Invalid_Reason"))
                          )

In [ ]:
quarantine_product_df.show()

#v. Quarantine Quantity

In [ ]:
quarantine_quantity_df = (silver_orders_df
                          .filter(~sf.col("valid_Quantity"))
                          .select(sf.col("Order_ID").alias("Order_ID"),
                                  sf.lit("Quantity").alias("Column_Name"),
                                  sf.col("raw_Quantity").alias("Invalid_Value"),
                                  sf.lit("DQ015").alias("Rule_ID"),
                                  sf.lit("Invalid Quantity format").alias("Invalid_Reason"))
                          )

In [ ]:
quarantine_quantity_df.show()

#vi. Quarantine Unit_Price

In [ ]:
quarantine_unit_price_df = (silver_orders_df
                          .filter(~sf.col("valid_Unit_Price"))
                          .select(sf.col("Order_ID").alias("Order_ID"),
                                  sf.lit("Unit_Price").alias("Column_Name"),
                                  sf.col("raw_Unit_Price").alias("Invalid_Value"),
                                  sf.lit("DQ016").alias("Rule_ID"),
                                  sf.lit("Invalid Unit_Price format").alias("Invalid_Reason"))
                          )

In [ ]:
quarantine_unit_price_df.show()

#vii. Qurantine Order_Status

In [ ]:
quarantine_order_status_df = (silver_orders_df
                          .filter(~sf.col("valid_Order_Status"))
                          .select(sf.col("Order_ID").alias("Order_ID"),
                                  sf.lit("Order_Status").alias("Column_Name"),
                                  sf.col("raw_Order_Status").alias("Invalid_Value"),
                                  sf.lit("DQ017").alias("Rule_ID"),
                                  sf.lit("Invalid Order_Status format").alias("Invalid_Reason"))
                          )

In [ ]:
quarantine_order_status_df.show()

#viii. Combined Quarantined Orders table

In [ ]:
quarantine_orders_df = (quarantine_order_id_df
                        .unionByName(quarantine_order_customers_df)
                        .unionByName(quarantine_order_date_df)
                        .unionByName(quarantine_product_df)
                        .unionByName(quarantine_quantity_df)
                        .unionByName(quarantine_unit_price_df)
                        .unionByName(quarantine_order_status_df)
                        )

In [ ]:
quarantine_orders_df.show()

#Step 5: Creating silver delta tables

#A. Creating silver delta path

In [ ]:
silver_delta_cleaned_customer_path = f"{project_root}/silver/cleaned/customer"
silver_delta_cleaned_orders_path = f"{project_root}/silver/cleaned/orders"
silver_delta_final_customer_path = f"{project_root}/silver/final/customer"
silver_delta_final_orders_path = f"{project_root}/silver/final/orders"
silver_delta_quarantine_customer_path = f"{project_root}/silver/quarantine/customer"
silver_delta_quarantine_orders_path = f"{project_root}/silver/quarantine/orders"

#B. Ectracting final silver tables

#i. Customers

In [ ]:
final_silver_customers_df = (silver_customers_df
                             .filter(~sf.col("Validation_Failed"))
                             .select("Customer_ID", "Customer_Name", "Email",
                                     "Phone", "City", "Registration_Date")
                             )

In [ ]:
final_silver_customers_df.show(truncate=False)

#ii. Orders

In [ ]:
final_silver_orders_df = (silver_orders_df
                          .filter(~sf.col("Validation_Failed"))
                          .select("Order_ID", "Customer_ID", "Order_Date", "Product",
                                  "Quantity", "Unit_Price", "Order_Status")
                          )

In [ ]:
final_silver_orders_df.show()

#C. Storing into delta

In [ ]:
silver_customers_df.write.format("delta").mode("overwrite").save(silver_delta_cleaned_customer_path)
silver_orders_df.write.format("delta").mode("overwrite").save(silver_delta_cleaned_orders_path)
final_silver_customers_df.write.format("delta").mode("overwrite").save(silver_delta_final_customer_path)
final_silver_orders_df.write.format("delta").mode("overwrite").save(silver_delta_final_orders_path)
quarantine_customers_df.write.format("delta").mode("overwrite").save(silver_delta_quarantine_customer_path)
quarantine_orders_df.write.format("delta").mode("overwrite").save(silver_delta_quarantine_orders_path)

#D. Reading back final silver data

In [ ]:
silver_delta_customers_df = spark.read.format("delta").load(silver_delta_final_customer_path)
silver_delta_orders_df = spark.read.format("delta").load(silver_delta_final_orders_path)

#Step 6: Inspecting final silver data

In [ ]:
silver_delta_customers_df.printSchema()
silver_delta_orders_df.printSchema()

In [ ]:
silver_delta_customers_df.show(truncate=False)
silver_delta_orders_df.show(truncate=False)

In [ ]:
print("Customers:", silver_delta_customers_df.count())
print("Orders:", silver_delta_orders_df.count())

print("Duplicate Customer IDs:",
      silver_delta_customers_df
      .groupBy("Customer_ID")
      .count()
      .filter(sf.col("count") > 1)
      .count())

print("Duplicate Order IDs:",
      silver_delta_orders_df
      .groupBy("Order_ID")
      .count()
      .filter(sf.col("count") > 1)
      .count())

#Step 8: Gold Layer Transformation

#A, Customer Sales Summary

In [ ]:
gold_customer_sales_summary_df = (silver_delta_customers_df
                                  .join(silver_delta_orders_df
                                        .filter(sf.col("Order_Status") != "CANCELLED"),
                                        on="Customer_ID", how="left")
                                  .groupBy("Customer_ID", "Customer_Name", "City",
                                           "Registration_Date")
                                  .agg(sf.coalesce(sf.count("Order_ID"), sf.lit(0))
                                  .alias("Number_of_Orders"),
                                       sf.coalesce(sf.sum(sf.col("Quantity") *
                                                          sf.col("Unit_Price")), sf.lit(0))
                                       .alias("Total_Amount"),
                                       sf.coalesce(sf.round(sf.avg(sf.col("Quantity") *
                                                       sf.col("Unit_Price")), 2), sf.lit(0))
                                       .alias("Average_Order_Amount"),
                                       sf.coalesce(sf.sum("Quantity"), sf.lit(0))
                                       .alias("Total_Quantity"))
                                  )

In [ ]:
gold_customer_sales_summary_df.show()

In [ ]:
gold_customer_sales_summary_df.printSchema()

print("Gold records:", gold_customer_sales_summary_df.count())

gold_customer_sales_summary_df.filter(
    (sf.col("Number_of_Orders") < 0) |
    (sf.col("Total_Amount") < 0) |
    (sf.col("Total_Quantity") < 0)
).show()

#Step 7: Wiring gold to delta

#A. Gold path

In [ ]:

gold_delta_customer_sales_path = f"{project_root}/gold/customer_sales_summary"

#B. Write into delta

In [ ]:
gold_customer_sales_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_delta_customer_sales_path)

#C. Reading back from delta

In [ ]:
gold_customer_sales_summary_delta_df = (spark.read.format("delta")
.load(gold_delta_customer_sales_path))

gold_customer_sales_summary_delta_df.show(truncate=False)

In [ ]:
print("Gold Delta records:", gold_customer_sales_summary_delta_df.count())

#D. Gold Layer Validation

#i. Gold Customer Coverage validation

In [ ]:
(final_silver_customers_df
 .join(gold_customer_sales_summary_delta_df.select("Customer_ID"),
       on="Customer_ID", how="left_anti")
 .select("Customer_ID")
 .show()
 )

#ii. Cancelled Orders Exclusion

In [ ]:
(final_silver_customers_df
 .join(final_silver_orders_df.filter(sf.col("Order_Status") == "CANCELLED"),
       on="Customer_ID", how="inner")
 .select("Customer_ID", "Order_ID")
 .show())

In [ ]:
(gold_customer_sales_summary_delta_df
 .filter(sf.col("Customer_ID") == "CUST1010")
 .select(
     "Customer_ID",
     "Number_of_Orders",
     "Total_Amount",
     "Average_Order_Amount",
     "Total_Quantity"
 )
 .show()
)

#E. Calculation Accuracy

In [ ]:
(gold_customer_sales_summary_delta_df
 .filter(sf.col("Customer_ID") == "CUST1001")
 .select(
     "Customer_ID",
     "Number_of_Orders",
     "Total_Amount",
     "Average_Order_Amount",
     "Total_Quantity"
 )
 .show()
)

#F. Zero-Order Customer Handling

In [ ]:
(gold_customer_sales_summary_delta_df
 .filter((sf.col("Number_of_Orders") == 0) &
  (
      (sf.col("Total_Amount") != 0) |
       (sf.col("Average_Order_Amount") != 0) |
        (sf.col("Total_Quantity") != 0)
        ))
 .select("Customer_ID", "Number_of_Orders","Total_Amount",
         "Average_Order_Amount", "Total_Quantity")
 .show()
)

#G. Metric Consistency

In [ ]:
(gold_customer_sales_summary_delta_df
 .filter(
     (sf.col("Number_of_Orders") > 0) &
     (sf.col("Total_Quantity") <= 0)
 )
 .select("Customer_ID", "Number_of_Orders", "Total_Quantity")
 .show()
)

#H. No Negative Metrics

In [ ]:
(gold_customer_sales_summary_delta_df
 .filter(
     (sf.col("Number_of_Orders") < 0) |
     (sf.col("Total_Amount") < 0) |
     (sf.col("Average_Order_Amount") < 0) |
     (sf.col("Total_Quantity") < 0)
 )
 .select("Customer_ID", "Number_of_Orders", "Total_Amount",
         "Average_Order_Amount", "Total_Quantity")
 .show()
)

#I. Source-to-Gold Amount Reconciliation

#i. Step 1 — Expected total from Silver

In [ ]:
expected_total = (final_silver_orders_df
                  .filter(sf.col("Order_Status") != "CANCELLED")
                  .withColumn("Order_Amount",
                              sf.col("Quantity") * sf.col("Unit_Price"))
                  .agg(sf.sum("Order_Amount").alias("expected_total"))
                  .first()["expected_total"]
                  )

print("Expected Total:", expected_total)

#ii. Step 2 — Total from Gold

In [ ]:
actual_total = (gold_customer_sales_summary_delta_df
                .agg(sf.sum("Total_Amount").alias("actual_total"))
                .first()["actual_total"]
                )

print("Gold Total:", actual_total)

#iii. Reconciliation:

In [ ]:
print("Reconciliation:", expected_total == actual_total)

#iv. Join-loss investigation

In [ ]:
matched_orders = (
    final_silver_orders_df
    .filter(sf.col("Order_Status") != "CANCELLED")
    .join(
        final_silver_customers_df
        .filter(sf.col("valid_Customer_ID"))
        .select("Customer_ID")
        .distinct(),
        on="Customer_ID",
        how="inner"
    )
    .withColumn(
        "Order_Amount",
        sf.col("Quantity") * sf.col("Unit_Price")
    )
)

print(
    "Matched Order Total:",
    matched_orders.agg(sf.sum("Order_Amount").alias("total")).first()["total"]
)

In [ ]:
print("Expected Total:", expected_total)
print("Gold Total:", actual_total)
print("Difference:", expected_total - actual_total)

In [ ]:
(final_silver_orders_df
 .filter(sf.col("Order_Status") != "CANCELLED")
 .withColumn(
     "Order_Amount",
     sf.col("Quantity") * sf.col("Unit_Price")
 )
 .select(
     "Order_ID",
     "Customer_ID",
     "Quantity",
     "Unit_Price",
     "Order_Status",
     "Order_Amount"
 )
 .orderBy(sf.col("Order_Amount").desc())
 .show(truncate=False)
)

In [ ]:
(final_silver_orders_df
 .filter(sf.col("Order_Status") != "CANCELLED")
 .filter(
     ~sf.col("Customer_ID").isin(
         final_silver_customers_df
         .filter(sf.col("valid_Customer_ID"))
         .select("Customer_ID")
         .distinct()
         .rdd.flatMap(lambda x: x)
         .collect()
     )
 )
 .select(
     "Order_ID",
     "Customer_ID",
     "Quantity",
     "Unit_Price",
     "Order_Status",
     (sf.col("Quantity") * sf.col("Unit_Price")).alias("Order_Amount")
 )
 .show(truncate=False)
)

In [ ]:
print("Final Silver Orders:", final_silver_orders_df.count())
print("Persisted Silver Orders:", silver_delta_orders_df.count())

print("Orders present in Final Silver but missing from Persisted Silver:")

(final_silver_orders_df
 .select("Order_ID", "Customer_ID", "Order_Status", "Quantity", "Unit_Price")
 .exceptAll(
     silver_delta_orders_df
     .select("Order_ID", "Customer_ID", "Order_Status", "Quantity", "Unit_Price")
 )
 .show(truncate=False))

In [ ]:
print("Orders present in Persisted Silver but missing from Final Silver:")

(silver_delta_orders_df
 .select("Order_ID", "Customer_ID", "Order_Status", "Quantity", "Unit_Price")
 .exceptAll(
     final_silver_orders_df
     .select("Order_ID", "Customer_ID", "Order_Status", "Quantity", "Unit_Price")
 )
 .show(truncate=False))

## Project Conclusion

This project demonstrates an end-to-end Customer & Order Management ETL
pipeline using PySpark and Delta Lake.

Key capabilities implemented:

- Bronze-layer raw data ingestion
- Silver-layer data cleansing and standardization
- Field-level data validation
- Invalid-value handling
- Raw-value preservation
- Duplicate detection
- Referential integrity validation
- Business-rule validation
- Gold-layer customer sales aggregation
- Delta Lake persistence and read-back verification

The project follows a Bronze → Silver → Gold architecture and demonstrates
how data quality checks can be integrated throughout an ETL pipeline.